In [ ]:
!pip install transformers peft trl datasets faiss-cpu sentence-transformers tqdm torch bitsandbytes hf_xet bitsandbytes


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [15]:
import os
import torch
import numpy as np
import faiss
import re
import gc

embedding_path = "/content/embeddings.npy"
document_path = "/content/documents.txt"
faiss_index_path = "/content/faiss_index.index"

def reset_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def clean_text(t):
    return re.sub(r'\s+', ' ', t.strip())

def save(embs, docs):
    np.save(embedding_path, embs.cpu().numpy())
    with open(document_path, "w", encoding="utf-8") as f:
        f.writelines(f"{doc}\n" for doc in docs)

def load():
    if not os.path.exists(embedding_path):
        return None, None
    embs = torch.tensor(np.load(embedding_path))
    docs = open(document_path).read().splitlines()
    return embs, docs

def save_index(index):
    faiss.write_index(index, faiss_index_path)

def load_index():
    return faiss.read_index(faiss_index_path)

def build_index(embs):
    embs = embs.cpu().numpy().astype("float32")
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

def clean_and_overwrite_answer_file(file_path="/content/answer.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    qa_blocks = re.findall(r"Query: (.*?)\n+Answer: (.*?)(?=\n+Query:|\Z)", content, re.DOTALL)
    cleaned_output = ""
    for query, answer in qa_blocks:
        answer = answer.strip()
        sentences = answer.split('. ')
        if len(sentences) > 1 and sentences[-1][-1] not in '.!?':
            sentences.pop()
        cleaned_answer = '. '.join(sentences) + ('.' if sentences else '')
        cleaned_output += f"Question: {query.strip()}\nAnswer: {cleaned_answer.strip()}\n\n"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cleaned_output.strip())
    print(f"Answers cleaned and saved to: {file_path}")


In [12]:
import torch
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    AutoModelForCausalLM, BitsAndBytesConfig
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_encoder(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(device)
    return tokenizer, model

def load_reranker(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name).to(device)
    return tokenizer, model

def load_quantized_generator(model_name, for_training=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        quantization_config=quant_config,
        device_map="auto"
    )

    if for_training:
        model.gradient_checkpointing_enable()
        model = prepare_model_for_kbit_training(model)
        lora_config = LoraConfig(
            r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, lora_config)

    return tokenizer, model

def encode_query(query, tokenizer, model, max_query_length):
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=max_query_length).to(device)
    with torch.no_grad():
        embeddings = model.base_model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

def rerank(query, candidates, tokenizer, model):
    inputs = [tokenizer(query, doc, return_tensors="pt", padding=True, truncation=True).to(device) for doc in candidates]
    scores = [model(**input).logits.softmax(dim=-1).max().item() for input in inputs]
    ranked = [doc for _, doc in sorted(zip(scores, candidates), reverse=True)]
    return ranked

def generate_answer(query, context, tokenizer, model, max_new_tokens, temperature, top_p):
    input_text = f"Question: {query}\nContext: {context}\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)
    model.config.pad_token_id = model.config.eos_token_id
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[1].strip()


In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import os

top_k = 30
docs_to_embed = 5000
batch_size = 8
max_query_length = 256
max_new_tokens = 150
temperature = 0.7
top_p = 0.9
answer_path = "/content/answer.txt"

encoder_model_name = "BAAI/bge-base-en-v1.5"
reranker_model_name = "BAAI/bge-reranker-large"
generator_model_name = "deepcogito/cogito-v1-preview-llama-3B"

queries = [
    "What are the main components of the Earth's atmosphere?",
    "How do black holes form?",
    "What are the advantages of renewable energy sources?",
    "Can you explain the theory of relativity in layman's terms?",
    "What is the significance of the Fibonacci sequence in nature?"
]

# Load models
gen_tok, gen_model = load_quantized_generator(generator_model_name)
enc_tok, enc_model = load_encoder(encoder_model_name)
rr_tok, rr_model = load_reranker(reranker_model_name)

# Load or compute embeddings
embs, docs = load()
if embs is None or docs is None:
    wiki = load_dataset("wikipedia", "20220301.en", split=f"train[:{docs_to_embed}]", trust_remote_code=True)
    docs = [clean_text(example["text"]) for example in wiki]
    all_embeddings = []
    for i in tqdm(range(0, len(docs), batch_size)):
        batch = docs[i:i+batch_size]
        inputs = enc_tok(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            outputs = enc_model(**inputs)
            batch_embs = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(batch_embs.cpu())
        reset_memory()
    embs = torch.cat(all_embeddings, dim=0)
    save(embs, docs)

# Build FAISS index
if os.path.exists(faiss_index_path):
    index = load_index()
else:
    index = build_index(embs)
    save_index(index)

# Run RAG
with open(answer_path, "w") as f:
    for q in queries:
        q_emb = encode_query(q, enc_tok, enc_model, max_query_length)
        _, idxs = index.search(q_emb.cpu().numpy(), top_k)
        candidates = [docs[i] for i in idxs[0]]
        reranked = rerank(q, candidates, rr_tok, rr_model)
        context = " ".join(reranked)[:2048]
        ans = generate_answer(q, context, gen_tok, gen_model, max_new_tokens, temperature, top_p)
        f.write(f"Query: {q}\n\nAnswer: {ans}\n\n")

clean_and_overwrite_answer_file()
print("Answers saved and cleaned in:", answer_path)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
def read_answer_file():
    with open("/content/answer.txt", "r", encoding="utf-8") as f:
        answer_content = f.read()
    return answer_content

answer_text = read_answer_file()
print(answer_text)


Question: What is the process of photosynthesis?
Answer: Photosynthesis is a process by which plants, algae, and some bacteria convert light energy, usually from the sun, into chemical energy in the form of glucose and other organic compounds. It involves the capture of light energy by pigments (like chlorophyll), the conversion of light energy into chemical energy (ATP and NADPH), and the synthesis of glucose from carbon dioxide and water using the energy captured. The overall process is represented by the chemical equation: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2 + light energy.
